In [1]:
TICKER = "AAPL"
BENCHMARK = "^GSPC"
START = "2015-01-01"

In [2]:
import yfinance as yf
import pandas as pd

raw = yf.download([TICKER, BENCHMARK], start=START, auto_adjust=True, progress=False)
close = raw["Close"]
close.head()

Ticker,AAPL,^GSPC
Date,,
2015-01-02,24.171762,2058.199951
2015-01-05,23.490797,2020.579956
2015-01-06,23.493010,2002.609985
2015-01-07,23.822432,2025.900024
2015-01-08,24.737738,2062.139893


In [3]:
prices = (
    close.reset_index()
         .melt(id_vars="Date", var_name="ticker", value_name="close")
         .rename(columns={"Date": "date"})
         .dropna()
)
prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")
prices.head()

,date,ticker,close
0,2015-01-02,AAPL,24.171762
1,2015-01-05,AAPL,23.490797
2,2015-01-06,AAPL,23.493010
3,2015-01-07,AAPL,23.822432
4,2015-01-08,AAPL,24.737738


In [4]:
import sqlite3
from pathlib import Path

DB_PATH = Path.cwd().parent / "data" / "prices.db"

with sqlite3.connect(DB_PATH) as conn:
    prices.to_sql("prices", conn, if_exists="replace", index=False)

In [8]:
with sqlite3.connect(DB_PATH) as conn:
    check = pd.read_sql("SELECT ticker, COUNT(*) AS n FROM prices GROUP BY ticker", conn)
check

,ticker,n
0,AAPL,2935
1,^GSPC,2935


In [9]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path.cwd().parent / "data" / "prices.db"

def q(sql):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql(sql, conn)

In [11]:
q("""
SELECT
date,
ticker,
close
FROM prices
WHERE ticker = 'AAPL' AND date >= '2024-01-01'
ORDER BY date
LIMIT 5
""")

,date,ticker,close
0,2024-01-02,AAPL,183.403992
1,2024-01-03,AAPL,182.030762
2,2024-01-04,AAPL,179.718933
3,2024-01-05,AAPL,178.997742
4,2024-01-08,AAPL,183.324982


In [12]:
q("""SELECT
date,
ticker,
close
FROM prices
ORDER BY date DESC
LIMIT 5""")

,date,ticker,close
0,2026-09-03,AAPL,328.209991
1,2026-09-03,^GSPC,7747.709961
2,2026-09-02,AAPL,324.959991
3,2026-09-02,^GSPC,7666.600098
4,2026-09-01,AAPL,325.130005


In [13]:
q("""
SELECT
    ticker,
    COUNT(*)      AS n_days,
    ROUND(AVG(close), 2) AS avg_close,
    ROUND(MIN(close), 2) AS min_close,
    ROUND(MAX(close), 2) AS max_close
FROM prices
GROUP BY ticker
""")

,ticker,n_days,avg_close,min_close,max_close
0,AAPL,2935,116.33,20.55,339.79
1,^GSPC,2935,3820.97,1829.08,7798.99


In [14]:
q("""
SELECT
    a.date,
    a.close AS aapl_close,
    b.close AS benchmark_close
FROM prices a
JOIN prices b
    ON a.date = b.date
WHERE a.ticker = 'AAPL' AND b.ticker = '^GSPC'
ORDER BY a.date
LIMIT 5
""")

,date,aapl_close,benchmark_close
0,2015-01-02,24.171762,2058.199951
1,2015-01-05,23.490797,2020.579956
2,2015-01-06,23.493010,2002.609985
3,2015-01-07,23.822432,2025.900024
4,2015-01-08,24.737738,2062.139893


In [15]:
returns = q("""
SELECT
    date,
    ticker,
    close,
    LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS prev_close,
    ROUND(
        (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date))
        / LAG(close) OVER (PARTITION BY ticker ORDER BY date),
        6
    ) AS daily_return
FROM prices
ORDER BY ticker, date
""")

returns.head(3)

,date,ticker,close,prev_close,daily_return
0,2015-01-02,AAPL,24.171762,NaN,NaN
1,2015-01-05,AAPL,23.490797,24.171762,-0.028172
2,2015-01-06,AAPL,23.493010,23.490797,0.000094


In [16]:
returns.dropna().head(3)

,date,ticker,close,prev_close,daily_return
1,2015-01-05,AAPL,23.490797,24.171762,-0.028172
2,2015-01-06,AAPL,23.493010,23.490797,0.000094
3,2015-01-07,AAPL,23.822432,23.493010,0.014022


In [17]:
with sqlite3.connect(DB_PATH) as conn:
    returns.dropna().to_sql("returns", conn, if_exists="replace", index=False)

q("SELECT ticker, COUNT(*) AS n FROM returns GROUP BY ticker")

,ticker,n
0,AAPL,2934
1,^GSPC,2934


In [18]:
returns = q("SELECT date, ticker, daily_return FROM returns")

wide = returns.pivot(index="date", columns="ticker", values="daily_return").dropna()
wide.head()

ticker,AAPL,^GSPC
date,,
2015-01-05,-0.028172,-0.018278
2015-01-06,0.000094,-0.008893
2015-01-07,0.014022,0.011630
2015-01-08,0.038422,0.017888
2015-01-09,0.001073,-0.008404


In [19]:
import numpy as np

TRADING_DAYS = 252

daily_vol = wide["AAPL"].std()
annual_vol = daily_vol * np.sqrt(TRADING_DAYS)

print(f"Tägliche Vola: {daily_vol:.4%}")
print(f"Annualisierte Vola: {annual_vol:.4%}")

Tägliche Vola: 1.8107%
Annualisierte Vola: 28.7432%


In [21]:
RISK_FREE_ANNUAL = 0.04 #US-Anleihe

daily_rf = RISK_FREE_ANNUAL / TRADING_DAYS
excess_daily = wide["AAPL"] - daily_rf

sharpe = (excess_daily.mean() / excess_daily.std()) * np.sqrt(TRADING_DAYS)
print(f"Sharpe Ratio: {sharpe:.2f}")

Sharpe Ratio: 0.78


In [22]:
cumulative = (1 + wide["AAPL"]).cumprod()
running_max = cumulative.cummax()
drawdown = cumulative / running_max - 1

max_drawdown = drawdown.min()
print(f"Max Drawdown: {max_drawdown:.2%}")

Max Drawdown: -38.52%


In [23]:
import statsmodels.api as sm

y = wide["AAPL"]
X = sm.add_constant(wide["^GSPC"])

model = sm.OLS(y, X).fit()
beta = model.params["^GSPC"]
alpha = model.params["const"]

print(f"Beta: {beta:.2f}")
print(f"Alpha (täglich): {alpha:.4%}")
print(model.summary())

Beta: 1.19
Alpha (täglich): 0.0444%
                            OLS Regression Results                            
Dep. Variable:                   AAPL   R-squared:                       0.532
Model:                            OLS   Adj. R-squared:                  0.532
Method:                 Least Squares   F-statistic:                     3329.
Date:                Fri, 04 Sep 2026   Prob (F-statistic):               0.00
Time:                        23:11:40   Log-Likelihood:                 8720.0
No. Observations:                2934   AIC:                        -1.744e+04
Df Residuals:                    2932   BIC:                        -1.742e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0

In [25]:
summary = pd.DataFrame({
    "Kennzahl": ["Annualisierte Vola", "Sharpe Ratio", "Max Drawdown", "Beta"],
    "Wert": [f"{annual_vol:.2%}", f"{sharpe:.2f}", f"{max_drawdown:.2%}", f"{beta:.2f}"],
})
print(summary)


             Kennzahl     Wert
0  Annualisierte Vola   28.74%
1        Sharpe Ratio     0.78
2        Max Drawdown  -38.52%
3                Beta     1.19
